# Plate 05 · End-to-end Pipeline Evaluation

Measures what the worker actually experiences: **frame → detector → best-3 crops by sharpness → recogniser → vote**.

Targets (spec section 11): **95 % exact match by day, 90 % at night**, < 1.5 s on device.
Report separately: day / night, plate type, distance, and the share sent to the server fallback (confidence < 0.85).

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import time
from collections import Counter
import cv2, numpy as np, pandas as pd, torch
from ultralytics import YOLO
from Common_Utils.Data_Utils import variance_of_laplacian
from Common_Utils.Metrics_Utils import exact_match, character_error_rate
from Common_Utils.Plate_Normalizer import normalize

detector = YOLO(str(RUNS / "Plate_Detector" / "yolo11n_v1" / "weights" / "best.pt"))
CONF_THRESHOLD = 0.85

In [ ]:
def recognise_sequence(frames, recogniser):
    """frames: list of BGR images sampled every 150 ms (spec 11.1). recogniser(crops) -> (texts, confs)."""
    crops = []
    for frame in frames:
        result = detector.predict(frame, imgsz=640, conf=0.4, verbose=False)[0]
        if len(result.boxes) == 0:
            continue
        x1, y1, x2, y2 = result.boxes.xyxy[0].int().tolist()
        crop = frame[y1:y2, x1:x2]
        crops.append((variance_of_laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)), crop))
    best = [c for _, c in sorted(crops, key=lambda t: -t[0])[:3]]
    if not best:
        return None, 0.0
    texts, confs = recogniser(best)
    votes = Counter(normalize(t) for t in texts)
    winner, count = votes.most_common(1)[0]
    conf = float(np.mean([c for t, c in zip(texts, confs) if normalize(t) == winner]))
    return winner, conf * count / len(texts)

In [ ]:
# Evaluate on a held-out CSV: video_or_folder, true_text, condition (day/night), plate_type
EVAL_CSV = DATA / "Plates" / "Eval_Sequences.csv"
if EVAL_CSV.exists():
    df = pd.read_csv(EVAL_CSV)
    # df["pred"], df["conf"], df["latency_ms"] = ...  (run recognise_sequence per row)
    for cond, part in df.groupby("condition"):
        print(cond, "exact:", exact_match(part["pred"].tolist(), part["true_text"].map(normalize).tolist()))
else:
    print("Create", EVAL_CSV, "from consented field recordings first.")